# RF-DETR on Apple Silicon — Export, Inference & Latency

Compares every export path built for **Apple Silicon**: eager PyTorch (Metal via MPS, as the
reference anchor) against native CoreML, Apple's newer Core AI framework, and ExecuTorch's CoreML
delegate — each exported, loaded through the one public prediction API, `RFDETRInference`, and
benchmarked across a batch-size matrix.

| Format | Export | Inference |
|--------|--------|-----------|
| **PyTorch** | *(no export — eager model)* | `predict()` on `device="mps"` |
| **CoreML** | `model.export(format="coreml", batch_size=N)` → `.mlpackage` | `RFDETRInference(path).predict()` |
| **Core AI** | `model.export(format="coreai", batch_size=N)` → `.aimodel` | `RFDETRInference(path).predict()` |
| **ExecuTorch (CoreML backend)** | `model.export(format="executorch", backend="coreml", batch_size=N)` → `.pte` | `RFDETRInference(path).predict()` |

> **Not the same as native CoreML.** ExecuTorch's `coreml` backend is a different path from
> `format="coreml"` — it produces a `.pte` file for the ExecuTorch runtime, not a `.mlpackage`.
> It runs fp16 on the Neural Engine; native CoreML and Core AI default to fp32.

> **Local macOS only, Apple Silicon, macOS 27+.** Running (not just building) a CoreML or Core AI
> model requires their runtimes, which only exist on macOS / iOS / iPadOS. Core AI additionally
> needs macOS 27 or later. This notebook will not work on Colab or any Linux runner.

> **Not covered here**: NVIDIA GPU deployment (see the [CUDA cookbook](../export-cuda/)),
> general-purpose desktop/server CPU (see the [CPU cookbook](../export-cpu/)), or mobile/edge formats
> — TFLite, LiteRT, ExecuTorch's XNNPACK backend (see the [mobile cookbook](../export-mobile/)).
> ExecuTorch's `qnn` (Qualcomm) backend is Apple-irrelevant and covered nowhere in this series.

> **Which latency number is this?** Every number in this notebook is a **Tier C** measurement:
> end-to-end `predict()` timings (preprocess + forward + postprocess) for each export format, at batch sizes 1/4/8, on one Apple
> Silicon Mac, against a PyTorch (MPS) anchor measured on that same machine. Its value is *ratios
> within this machine* — never set these milliseconds against the architecture headline on the
> [Benchmarks page](https://rfdetr.roboflow.com/latest/learn/benchmarks/) (NVIDIA T4, TensorRT
> FP16, model only) or the number on
> [Roboflow's RF-DETR model page](https://docs.roboflow.com/models/supported-models/rf-detr)
> (L4, Roboflow Inference, scope not stated, batch 1) — those are GPU numbers, not comparable to an Apple
> Silicon run at all, let alone without checking precision, batch size, and scope.

## 1. Install

`RFDETR_SPEC` holds the install source and defaults to the `develop` branch. `RFDETRInference` and the
`rfdetr.export.benchmark` helpers this notebook imports ship in 1.12; to install from PyPI instead, swap in the
commented `>=1.12,<2` line once it is released.
`coremltools` (pulled in by
`[coreml]`) also satisfies ExecuTorch's CoreML backend, which needs it but does not declare it as
a dependency of `[executorch]`.

In [ ]:
import sys

assert (3, 11) <= sys.version_info[:2] < (3, 14), "Install and run this cookbook with Python 3.11, 3.12, or 3.13."

In [ ]:
RFDETR_SPEC = "rfdetr[coreml,coreai,executorch] @ https://github.com/roboflow/rf-detr/archive/refs/heads/develop.zip"
# RFDETR_SPEC = "rfdetr[coreml,coreai,executorch]>=1.12,<2"
!pip install -q "{RFDETR_SPEC}" psutil supervision pandas tabulate faster-coco-eval

## 2. Setup

Every format in this notebook needs Apple Silicon.
`predict()`'s postprocessing always ends by moving results to the CPU (`Detections` is numpy, not
a device tensor), which forces a full MPS stream sync — so timing `predict()` with a plain
wall-clock is already correct, the same way `.cpu()` forces a CUDA sync.

The measurement helpers come from the public `rfdetr.export.benchmark` module, shared with every
other per-hardware cookbook: `measure_latency` times a callable after warmup, and `measure_memory`
measures the host resident-memory growth of constructing a runtime and running its first
inference call. The installed `rfdetr` version is
printed so the Results header can be filled from it.

In [ ]:
import importlib.metadata
import platform
from functools import partial
from pathlib import Path

import torch
from PIL import Image

from rfdetr import RFDETRInference
from rfdetr.export._benchmark import _artifact_size_mb, visualize_detections
from rfdetr.export.benchmark import measure_latency, measure_memory

print(f"rfdetr: {importlib.metadata.version('rfdetr')}")

### Notebook display helpers

`_artifact_size_mb` and `visualize_detections` come from the private `rfdetr.export._benchmark` module, next to the
COCO mAP helpers used later. They size artifacts and draw detections; neither changes what a runtime computes.

In [ ]:
# Inline figures keep every plot in the notebook; closing them after display stops figures piling up in memory.
%matplotlib inline
%config InlineBackend.close_figures = True

`BATCH_SIZES = (1, 4, 8)` is the batch matrix every format below is benchmarked at. `rows` accumulates one record per
(format, config, batch) triple; `record()` appends it, and the Results section turns `rows` into the summary tables.
`artifact_paths` remembers each export's batch-1 artifact so the COCO section scores exactly the files that were timed.

In [ ]:
if platform.system() != "Darwin" or platform.machine() != "arm64":
    raise RuntimeError("This notebook requires Apple Silicon (macOS, arm64); none is available here.")
print(f"macOS {platform.mac_ver()[0]}, {platform.machine()}")

# Core AI's runtime ships with macOS 27. On an older macOS the same package loads and runs, but on the CPU only
# (measured ~3.6 s per call, identical to device="cpu"), so its rows would be wrong. Sections 7's cells are skipped there.
COREAI_RUNTIME_AVAILABLE = int(platform.mac_ver()[0].split(".")[0]) >= 27
print(f"Core AI runtime usable: {COREAI_RUNTIME_AVAILABLE}")

PYTORCH_DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"PyTorch device: {PYTORCH_DEVICE}")

EXPORT_DIR = Path("export_apple")
EXPORT_DIR.mkdir(exist_ok=True)
CONFIDENCE_THRESHOLD = 0.5
WARMUP_RUNS = 5
MEASURE_RUNS = 30
BATCH_SIZES = (1, 4, 8)

#: One record per (format, config, batch) benchmark, appended to by `record()`.
rows: list[dict] = []
#: Batch-1 artifact of every export, keyed by (Format, Config) — the files the COCO section scores.
artifact_paths: dict[tuple[str, str], Path] = {}


def record(format_label: str, config: str, batch: int, result, memory_mb: float, size_mb: float | None = None) -> None:
    """Append one benchmark result; `FPS` counts images per second, so it compares across batch sizes."""
    rows.append(
        {
            "Format": format_label,
            "Config": config,
            "Batch": batch,
            "end2end [ms]": result.mean_ms,
            "± [ms]": result.std_ms,
            "FPS [img/s]": batch * result.fps,
            "Memory [MB]": memory_mb,
            "Size [MB]": size_mb,
        }
    )

## 3. Sample image

A single street scene with several COCO classes is enough to verify detections. The image is downloaded once and reused for every format below.

In [ ]:
import urllib.request

IMAGE_URL = "https://media.roboflow.com/notebooks/examples/dog.jpeg"
IMAGE_PATH = EXPORT_DIR / "sample.jpg"
if not IMAGE_PATH.exists():
    urllib.request.urlretrieve(IMAGE_URL, IMAGE_PATH)

image = Image.open(IMAGE_PATH).convert("RGB")
print(f"Sample image: {image.size[0]}×{image.size[1]}")

## 4. PyTorch baseline — `predict()` on Metal (MPS)

No export needed — this is the reference every other format in this notebook is compared
against. `predict()` includes preprocessing and postprocessing — there is no separate
"forward-only" path through it, so only an end-to-end number is reported for the baseline.
`predict()` also accepts a list of images and batches them internally, so the batch matrix below
feeds it `batch` copies of the same sample image rather than exporting anything.

In [ ]:
from rfdetr import RFDETRSmall

with measure_memory() as mem:
    model = RFDETRSmall(device=PYTORCH_DEVICE)
    baseline_detections = model.predict(image, threshold=CONFIDENCE_THRESHOLD)
pytorch_baseline_memory_mb = mem.delta_mb
print(f"PyTorch baseline ({PYTORCH_DEVICE}): {len(baseline_detections)} detections above {CONFIDENCE_THRESHOLD}")
visualize_detections(baseline_detections, image)

`Memory [MB]` for PyTorch is measured once above (constructing the runtime is a one-time cost,
unlike the fixed-batch formats below which reload a fresh artifact per batch) and reused for
every batch row in the summary table.

In [ ]:
for batch in BATCH_SIZES:
    images_batch = [image] * batch
    pytorch_baseline = measure_latency(
        partial(model.predict, images_batch),
        label=f"PyTorch predict() ({PYTORCH_DEVICE}) batch={batch}",
        device="cpu",
        warmup=WARMUP_RUNS,
        runs=MEASURE_RUNS,
    )
    print(
        f"  {pytorch_baseline.label:<40}  {pytorch_baseline.mean_ms:6.2f} ms ± {pytorch_baseline.std_ms:5.2f}"
        f"   ({batch * pytorch_baseline.fps:6.1f} FPS)"
    )
    record("PyTorch predict()", f"{PYTORCH_DEVICE}, fp32", batch, pytorch_baseline, pytorch_baseline_memory_mb)

### PyTorch `inference(dtype=torch.float16)` — fp16+JIT anchor

`model.inference(dtype=torch.float16, batch_size=N)` returns an independent TorchScript
predictor compiled for exactly `batch_size` images — calling `predict()` with a different batch
afterward raises `ValueError: Batch size mismatch` (confirmed by an earlier run of this notebook).
Each batch gets its own predictor; deleting it releases that runtime and leaves `model` ready for
the export calls that follow.

In [ ]:
for batch in BATCH_SIZES:
    images_batch = [image] * batch
    with measure_memory() as mem:
        predictor = model.inference(dtype=torch.float16, batch_size=batch)
        _ = predictor.predict(images_batch)
    pytorch_fp16_jit_memory_mb = mem.delta_mb

    pytorch_fp16_jit = measure_latency(
        partial(predictor.predict, images_batch),
        label=f"PyTorch inference() fp16+JIT ({PYTORCH_DEVICE}) batch={batch}",
        device="cpu",
        warmup=WARMUP_RUNS,
        runs=MEASURE_RUNS,
    )
    print(
        f"  {pytorch_fp16_jit.label:<40}  {pytorch_fp16_jit.mean_ms:6.2f} ms ± {pytorch_fp16_jit.std_ms:5.2f}"
        f"   ({batch * pytorch_fp16_jit.fps:6.1f} FPS)   +{pytorch_fp16_jit_memory_mb:.1f} MB"
    )
    record("PyTorch inference()", f"{PYTORCH_DEVICE}, fp16+JIT", batch, pytorch_fp16_jit, pytorch_fp16_jit_memory_mb)
    del predictor

## 5. Export, load, benchmark — one path for every format

Every export below goes through the same two public calls: `model.export(format=..., batch_size=N)` writes the artifact,
and `RFDETRInference(path)` loads it for prediction. The loader reads the artifact's own metadata, so the same
`predict()` that serves the PyTorch rows above also serves CoreML, Core AI and ExecuTorch: preprocessing, postprocessing,
class ids and `sv.Detections` output are shared, and no per-format tensor names, dtypes, decoders or async wrappers are
needed.

`benchmark_export` exports one artifact per batch size, because CoreML, Core AI and ExecuTorch all bake a fixed batch
shape into the file. For each artifact it loads a predictor, runs it once on the sample image (batch 1 prints the
detection count and draws the result), and times `predict()` on `batch` copies of the sample. `Memory [MB]` is the host
resident-memory growth across constructing the predictor plus its first call.

Only end-to-end `predict()` timings are reported: the PyTorch rows have no forward-only path, so one scope keeps every
row of the table comparable. Correctness is checked by the COCO mAP section below, which scores these same batch-1
artifacts, instead of by comparing raw tensors against a reference runtime.

In [ ]:
def benchmark_export(format_label: str, config: str, subdir: str, **export_kwargs) -> None:
    """Export one artifact per batch size, load each with `RFDETRInference` and time its `predict()`."""
    for batch in BATCH_SIZES:
        path = Path(model.export(batch_size=batch, output_dir=str(EXPORT_DIR / f"{subdir}_b{batch}"), **export_kwargs))
        size_mb = _artifact_size_mb(path)
        images_batch = [image] * batch
        with measure_memory() as mem:
            predictor = RFDETRInference(path)
            detections = predictor.predict(images_batch, threshold=CONFIDENCE_THRESHOLD)
        if batch == 1:
            artifact_paths[(format_label, config)] = path
            print(f"{format_label} ({config}): {path}  ({size_mb:.1f} MB)")
            print(f"  {len(detections[0])} detections above {CONFIDENCE_THRESHOLD}; runtime: {predictor.runtime_info}")
            visualize_detections(detections[0], image, EXPORT_DIR / f"annotated_{subdir}.jpg")
        result = measure_latency(
            partial(predictor.predict, images_batch),
            label=f"{format_label} {config} batch={batch}",
            device="cpu",
            warmup=WARMUP_RUNS,
            runs=MEASURE_RUNS,
        )
        print(f"  batch={batch}: {result.mean_ms:6.2f} ms ± {result.std_ms:5.2f}   ({batch * result.fps:6.1f} FPS)")
        record(format_label, config, batch, result, mem.delta_mb, size_mb)
        del predictor

## 6. Native CoreML

**What it is.** Native CoreML export produces a `.mlpackage` you can drag directly into Xcode —
no ONNX, no ExecuTorch runtime involved. **Good for** Apple-native (iOS/macOS) app development:
it's the lowest-friction path into Apple's own ML stack, distinct from ExecuTorch's `coreml`
*backend* below (a different export path that produces a `.pte`, not a `.mlpackage`). See the
[CoreML export docs](https://rfdetr.roboflow.com/exports/coreml/).

### CoreML (fp32, default)

Export defaults to `coreml_precision="float32"` for tight CPU parity with eager PyTorch. The `.mlpackage` is loaded with
coremltools' default `ComputeUnit.ALL`; the printed `runtime_info` shows what `RFDETRInference` selected.

In [ ]:
benchmark_export("CoreML", "fp32 (default)", "coreml", format="coreml")

### CoreML (fp16)

`coreml_precision="float16"` halves the weight precision, producing a smaller, ANE-oriented
bundle. The tradeoff is larger numeric drift: fp16 narrows the fp32 safety margin around
near-tied `torch.topk` query-ranking decisions, so a rank swap (and therefore a slightly
different detection set) becomes more likely than in the fp32 row above. The mAP section below measures it.

In [ ]:
benchmark_export("CoreML", "fp16", "coreml_fp16", format="coreml", coreml_precision="float16")

## 7. Core AI

**What it is.** [Core AI](https://developer.apple.com/documentation/coreai) is Apple's newer
on-device inference framework (iOS, iPadOS, macOS 27+). RF-DETR traces the model with
`torch.export` and converts it to an `.aimodel` asset — no ONNX step. **Good for** apps
targeting the newest Apple OS versions: the runtime decides at load time whether the CPU, GPU,
or Neural Engine executes it, rather than that choice being baked in at export time like CoreML.
See the [Core AI export docs](https://rfdetr.roboflow.com/exports/coreai/).

### Core AI (fp32, default)

The `.aimodel` bakes a fixed batch shape into the asset, so `benchmark_export` writes one per batch size. Core AI's
Python runtime is partly async, but `RFDETRInference` hides that: `predict()` is an ordinary synchronous call, also
inside a notebook's running event loop. With the default specialization the runtime allows the CPU, GPU and Neural
Engine and sets no preferred unit, so the choice is its own, made at load time.

In [ ]:
if COREAI_RUNTIME_AVAILABLE:
    benchmark_export("Core AI", "fp32 (default)", "coreai", format="coreai")

### Core AI (fp16)

`coreai_precision="float16"` traces and stores the whole graph in fp16, so the asset takes and returns float16
tensors; `RFDETRInference` casts the input and the outputs, so the call site is unchanged. One part of the graph stays
fp32 regardless: RF-DETR forces `topk` back to float32 even in an fp16 export, because the Neural Engine's fp16 `topk`
returns corrupt indices and the two-stage query selection would gather the wrong encoder tokens. That caps how much of
the graph actually runs in fp16.

In [ ]:
if COREAI_RUNTIME_AVAILABLE:
    benchmark_export("Core AI", "fp16", "coreai_fp16", format="coreai", coreai_precision="float16")

## 8. ExecuTorch (CoreML backend)

**What it is.** This is the same ExecuTorch runtime as the [mobile cookbook](../export-mobile/)'s
XNNPACK section, but lowered to Apple's `coreml` *backend* instead — still a portable `.pte`
file, no `.mlpackage`, but running on the Apple Neural Engine. **Good for** staying inside the
ExecuTorch/PyTorch deployment story while still reaching Apple's Neural Engine, when native
CoreML's separate export/runtime isn't a fit; the tradeoff is fp16 only, unlike native CoreML and
Core AI's fp32 default. See the
[ExecuTorch export docs](https://rfdetr.roboflow.com/exports/executorch/#coreml-backend-apple-neural-engine-fp16).

### ExecuTorch (CoreML backend, fp16)

`backend="coreml"` targets the Apple Neural Engine, iPhone/iPad/Mac, and runs fp16 — unlike native CoreML and Core AI
above, which default to fp32. This is the ExecuTorch *delegate*, not native CoreML: it produces a `.pte` for the
ExecuTorch runtime, distinct from `format="coreml"`. ExecuTorch also bakes a fixed batch shape into the `.pte`, so
`benchmark_export` writes one per batch size. `RFDETRInference` owns the runtime details the raw `Runtime` API leaves to
the caller: a contiguous input tensor, a fresh input buffer per call and the torch/ExecuTorch compatibility check.
rfdetr passes ExecuTorch's CoreML partitioner no compile specs, so the backend resolves compute units to its own default
when lowering.

In [ ]:
benchmark_export("ExecuTorch", "CoreML backend, fp16", "executorch", format="executorch", backend="coreml")

## 9. COCO val2017 accuracy

Latency alone hides a precision collapse: an export can run at a normal speed and still return
nothing useful (an OpenVINO f16 execution bug once gave 0 detections). So every batch-1 row of the
Results table below also gets a box mAP, measured on the **same exported file at the same
precision** that row timed — the batch-1 `.mlpackage`, `.aimodel`, and `.pte` the export loops wrote, and the
PyTorch model in the same mode.

This is a **500-image seeded subset** of COCO val2017, so it is not the full-val2017 number on the
[Benchmarks page](https://rfdetr.roboflow.com/latest/learn/benchmarks/); set `FULL_VAL = True` to
score all 5000 images. Each row costs roughly its batch-1 end2end latency × the image count — about
6–12 s per row on the M4 Max at 500 images, ten times that with `FULL_VAL`. The first run downloads
the 241 MB annotation archive once, plus only the selected images.

Every row is scored the same way: `evaluate_coco_map` calls `predict(image, threshold=0.001)` and reads the returned
`sv.Detections`, whose `class_id` values are COCO category IDs for a pretrained checkpoint. The 0.001 score floor keeps
the whole precision-recall curve. Because no per-format decoding is involved, an exported row differs from the PyTorch
row only by what the export itself changed.

Display and scoring helpers come from the private `rfdetr.export._benchmark` module; they are not a
supported API and do not change what any runtime computes.

In [ ]:
from rfdetr.export._benchmark import CocoMapResult, evaluate_coco_map, fetch_coco_val2017

N_EVAL = 500
FULL_VAL = False
COCO_SEED = 0
COCO_ROOT = Path("coco")
COCO_SCORE_FLOOR = 0.001  # keeps the whole precision-recall curve

#: One entry per batch-1 Results row, keyed by that row's (Format, Config).
map_results: dict[tuple[str, str], CocoMapResult] = {}

The annotations and the selected images land in `COCO_ROOT`; files already on disk are skipped, so
rerunning this cell after the first download costs nothing.

In [ ]:
coco_subset = fetch_coco_val2017(COCO_ROOT, n_images=None if FULL_VAL else N_EVAL, seed=COCO_SEED)
print(f"COCO val2017: {len(coco_subset.image_ids)} images in {coco_subset.images_dir}")

PyTorch first: `predict()` on MPS in fp32 is the eager reference every exported row is compared against, then the
fp16+JIT predictor at batch 1, the same `inference()` call as its latency row.

In [ ]:
map_results[("PyTorch predict()", f"{PYTORCH_DEVICE}, fp32")] = evaluate_coco_map(
    partial(model.predict, threshold=COCO_SCORE_FLOOR), coco_subset
)

predictor = model.inference(dtype=torch.float16, batch_size=1)
map_results[("PyTorch inference()", f"{PYTORCH_DEVICE}, fp16+JIT")] = evaluate_coco_map(
    partial(predictor.predict, threshold=COCO_SCORE_FLOOR), coco_subset
)
del predictor

Then every exported artifact, loaded exactly as in its latency row. A near-tied `topk` rank swap in an fp16 export
(see the CoreML fp16 note) shows up here as lost accuracy rather than only as a different detection set.

In [ ]:
for (format_label, config), path in artifact_paths.items():
    predictor = RFDETRInference(path)
    map_results[(format_label, config)] = evaluate_coco_map(
        partial(predictor.predict, threshold=COCO_SCORE_FLOOR), coco_subset
    )
    del predictor
    print(f"  {format_label} ({config}): mAP@50:95 {100 * map_results[(format_label, config)].map50_95:.1f} %")

## 9. Results

Measured on an Apple M4 Max (arm64), macOS 26.7.1, `rfdetr 1.12.0.dev0`, `RFDETRSmall`, torch 2.12.1, batches 1/4/8, 5 warmup + 30 timed runs per (format, config, batch). Numbers vary by Apple Silicon generation — rerun the cells above to get yours. Every latency is the end-to-end `predict()` call (preprocess + forward + postprocess) through `RFDETRInference`, or `predict()` itself for the PyTorch rows; `FPS` is images per second at that batch size, not a per-call rate. `mAP@50:95` is a 500-image seeded COCO val2017 subset, scored on the same batch-1 artifact that was timed. `Speed vs PyTorch ×` is the batch-1 latency of PyTorch `predict()` divided by the row's own.

| Format              | Config               |   b1 latency [ms] |   Speed vs PyTorch × |   Size [MB] |   mAP@50:95 [%] |   FPS b1 |   FPS b4 |   FPS b8 |
|:--------------------|:---------------------|------------------:|---------------------:|------------:|----------------:|---------:|---------:|---------:|
| PyTorch predict()   | mps, fp32            |              22.6 |                  1.0 |           — |            56.5 |     44.2 |     69.3 |     76.1 |
| PyTorch inference() | mps, fp16+JIT        |              16.0 |                  1.4 |           — |            56.5 |     62.3 |     83.0 |     83.6 |
| CoreML              | fp32 (default)       |              12.7 |                  1.8 |       114.7 |            56.5 |     78.9 |    113.6 |    121.3 |
| CoreML              | fp16                 |              24.4 |                  0.9 |        57.7 |            56.5 |     41.0 |     47.8 |     62.0 |
| ExecuTorch          | CoreML backend, fp16 |              24.4 |                  0.9 |        58.2 |            56.5 |     41.0 |     49.1 |     83.0 |

**Pick CoreML fp32 on this chip.** It is the fastest row at every batch size (12.7 ms at batch 1, 121 FPS at batch 8) with the same mAP as every other row. Its fp16 export is half the size but about twice as slow at batch 1 (24.4 vs 12.7 ms), so on Apple Silicon fp16 buys disk space, not speed; measure it per chip generation before relying on it. ExecuTorch's CoreML backend matches CoreML fp16 at batch 1 and scales better to batch 8 (83.0 vs 62.0 FPS), but stays behind CoreML fp32. All five rows score 56.5 mAP@50:95 on the 500-image subset, so on this model the choice is about latency, size and tooling, not accuracy.

**Core AI is not in this table.** Its runtime ships with macOS 27. On macOS 26.7.1 the `coreai` package loads and runs but only on the CPU (about 2.1 s per call for the raw runtime, for every compute-unit preference; `SpecializationOptions.is_supported()` returns `False`), so those rows would be wrong. The notebook skips them unless it runs on macOS 27 or later; rerun it there to fill them in.

> **Note:** the machine was not idle: another job ran alongside this measurement, so several `std` values (CoreML fp16 batch 8: ± 11.0 ms; ExecuTorch batch 8: ± 16.3 ms) are noisier than the batch-1 rows. Rerun on an idle machine for tighter comparisons. `Memory [MB]` (host RSS growth around each row's construct + first call, in one shared process) is an approximation: see the per-row values in the detail table printed below.

In [ ]:
import pandas as pd
from IPython.display import display

KEYS = ["Format", "Config"]
detail = pd.DataFrame(rows)

# Decision table: one row per (format, config); batch-1 latency, accuracy and size, throughput at every batch.
fps_by_batch = detail.pivot_table(index=KEYS, columns="Batch", values="FPS [img/s]", sort=False).add_prefix("FPS b")
map_by_row = pd.Series({key: 100 * result.map50_95 for key, result in map_results.items()}, name="mAP@50:95 [%]")
map_by_row.index.names = KEYS
headline = (
    detail[detail["Batch"] == 1]
    .set_index(KEYS)[["end2end [ms]", "Size [MB]"]]
    .rename(columns={"end2end [ms]": "b1 latency [ms]"})
    .join(map_by_row)
    .join(fps_by_batch)
)
headline.insert(1, "Speed vs PyTorch ×", headline["b1 latency [ms]"].iloc[0] / headline["b1 latency [ms]"])
display(headline.round(1))
print(headline.reset_index().fillna("—").to_markdown(index=False, floatfmt=".1f"))

# Full per-batch records, including ± std and memory growth, for anyone who wants the detail.
display(detail.round(2))
print(
    f"\n{MEASURE_RUNS} timed + {WARMUP_RUNS} warmup runs per row, batches {BATCH_SIZES}, {platform.machine()}, "
    f"macOS {platform.mac_ver()[0]}."
)

## Next steps

- **Fine-tuned weights** — pass `pretrain_weights="<path/to/checkpoint.pth>"` when constructing
  the model.
- **Smaller / faster bundle** — `coreml_precision="float16"` (CoreML) or `coreai_precision="float16"`
  (Core AI) trade numeric drift for size and, on some Apple Silicon generations, latency; see the
  [Core AI export docs](https://rfdetr.roboflow.com/exports/coreai/#precision-compute-units-and-latency)
  for measured per-device tradeoffs.
- **Deploy in Xcode** — drag the `.mlpackage` into your Xcode project, or load the `.aimodel` /
  `.pte` with the Core AI or ExecuTorch runtime for that platform.
- **Not on Apple Silicon?** — see the [CUDA cookbook](../export-cuda/), the
  [CPU cookbook](../export-cpu/), or the [mobile/edge cookbook](../export-mobile/).
- See the [Export documentation](https://rfdetr.roboflow.com/learn/export/) for every format and
  option.